# Archive historique — preparer_lot_test_independant

Ce notebook est conservé pour documenter la préparation et les évaluations antérieures. Il ne doit pas être exécuté pour régénérer les jeux de test déjà validés.

In [ ]:
"""Prépare un test indépendant et une file de relecture ciblée.

Le test est sélectionné parmi les PDF qui n'ont pas servi au lot pilote, à
l'échantillon de développement ni à la référence déjà validée. Après son
exécution, ne pas modifier les règles d'extraction avant sa validation humaine
complète et son évaluation.

Usage :
  python Scripts/pipeline_et_validation/archive_mise_au_point/preparer_lot_test_independant.ipynb
"""

from __future__ import annotations

import argparse
import csv
import hashlib
import json
import re
from collections import Counter, defaultdict
from pathlib import Path

def date_nom_fichier(pdf: Path) -> str:
    match = re.search(r"(20\d{2})_(\d{2})_(\d{2})", pdf.name)
    return f"{match[1]}-{match[2]}-{match[3]}" if match else ""


def lire_pdf(pdf: Path) -> tuple[str, str, str]:
    from io import BytesIO
    from pdfminer.high_level import extract_text
    texte = extract_text(str(pdf)) or ""
    if len(re.sub(r"\s+", "", texte)) >= 120:
        return texte, "pdfminer", ""
    import pymupdf
    import pytesseract
    from PIL import Image
    pages = []
    with pymupdf.open(pdf) as document:
        for page in document:
            pixmap = page.get_pixmap(matrix=pymupdf.Matrix(2, 2), alpha=False)
            image = Image.open(BytesIO(pixmap.tobytes("png")))
            pages.append(pytesseract.image_to_string(image, lang="fra"))
    return "\f".join(pages), "ocr_fra", ""


class pipeline:
    date_nom_fichier = staticmethod(date_nom_fichier)
    lire_pdf = staticmethod(lire_pdf)


def trouver_racine() -> Path:
    courant = Path.cwd().resolve()
    for candidat in (courant, *courant.parents):
        if (candidat / "total_donnees").is_dir() and (candidat / "evaluation_extraction").is_dir():
            return candidat
    raise FileNotFoundError("Ouvre ce notebook depuis le dossier de travail.")

RACINE = trouver_racine()
DOSSIER = RACINE / "evaluation_extraction"
TEST = DOSSIER / "test_independant"
COLONNES_REFERENCE = [
    "document_id", "page", "type_nomination", "date_conseil", "civilite", "nom",
    "matricule", "fonction", "ministere", "extrait_exact", "commentaire",
    "statut_verification", "mode_proposition",
]


def lire(chemin: Path) -> list[dict[str, str]]:
    with chemin.open(encoding="utf-8-sig", newline="") as fichier:
        return list(csv.DictReader(fichier))


def ecrire(chemin: Path, colonnes: list[str], lignes: list[dict[str, str]]) -> None:
    chemin.parent.mkdir(parents=True, exist_ok=True)
    with chemin.open("w", encoding="utf-8-sig", newline="") as fichier:
        writer = csv.DictWriter(fichier, fieldnames=colonnes, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(lignes)


def annee(identifiant: str) -> int:
    match = re.search(r"(20\d{2})", identifiant)
    return int(match.group(1)) if match else 0


def bande_annee(valeur: int) -> str:
    if valeur <= 2016:
        return "2010-2016"
    if valeur <= 2019:
        return "2017-2019"
    if valeur <= 2022:
        return "2020-2022"
    return "2023-2026"


def ordre_aleatoire(identifiant: str, graine: str) -> str:
    """Ordre stable, sans consulter les noms ou les extraits des annonces."""
    return hashlib.sha256(f"{graine}:{identifiant}".encode()).hexdigest()


def choisir_equilibre(candidats: list[dict[str, object]], nombre: int, graine: str) -> list[dict[str, object]]:
    """Choisit en alternant les bandes d'année, puis selon un ordre stable."""
    par_bande: dict[str, list[dict[str, object]]] = defaultdict(list)
    for candidat in candidats:
        par_bande[str(candidat["bande_annee"])].append(candidat)
    for valeurs in par_bande.values():
        valeurs.sort(key=lambda x: ordre_aleatoire(str(x["document_id"]), graine))
    resultat = []
    while len(resultat) < nombre:
        ajoute = False
        for bande in sorted(par_bande):
            if par_bande[bande] and len(resultat) < nombre:
                resultat.append(par_bande[bande].pop(0))
                ajoute = True
        if not ajoute:
            break
    return resultat


def empreinte() -> dict[str, str]:
    fichiers = [
        RACINE / "Scripts" / "pipeline_et_validation" / "pipeline_extraction_nominations.ipynb",
        RACINE / "Scripts" / "pipeline_et_validation" / "archive_mise_au_point" / "proposer_annotations.ipynb",
    ]
    return {fichier.name: hashlib.sha256(fichier.read_bytes()).hexdigest() for fichier in fichiers}


def risque(profil: dict[str, object]) -> tuple[int, str]:
    raisons = []
    score = 0
    if profil["methode_lecture"] != "pdfminer":
        score += 100
        raisons.append("OCR ou lecture dégradée")
    if profil["candidats"] == 0:
        score += 80
        raisons.append("aucune nomination candidate")
    if profil["a_revoir"]:
        score += min(40, 15 + int(profil["a_revoir"]))
        raisons.append("champs incomplets")
    if profil["conseils"]:
        score += 15
        raisons.append("conseil d'administration")
    if profil["candidats"] >= 100:
        score += 20
        raisons.append("liste longue")
    if profil["moyenne"]:
        score += 5
        raisons.append("contexte ou OCR")
    return score, "; ".join(raisons) or "contrôle aléatoire"


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--taille", type=int, default=28, help="Nombre indicatif de PDF du test.")
    parser.add_argument("--graine", default="memoire-burkina-2026-test-v1", help="Graine de sélection reproductible.")
    parser.add_argument("--minimum-candidats", type=int, default=300, help="Seuil indicatif de candidats dans le test.")
    parser.add_argument("--maximum-candidats", type=int, default=750, help="Charge maximale visée pour le test humain.")
    args = parser.parse_args()
    if args.taille < 12:
        raise ValueError("Le lot indépendant doit contenir au moins 12 PDF.")

    manifestes = lire(DOSSIER / "manifeste_pipeline_corpus.csv")
    extractions = lire(DOSSIER / "extractions_pipeline_corpus.csv")
    extractions_par_document: dict[str, list[dict[str, str]]] = defaultdict(list)
    for ligne in extractions:
        extractions_par_document[ligne["document_id"]].append(ligne)

    exclus = set()
    for nom in ("documents_echantillon.csv", "lot_pilote_12_documents.csv", "reference_annotations_validees.csv"):
        exclus.update(x["document_id"] for x in lire(DOSSIER / nom))

    profils = []
    for manifeste in manifestes:
        identifiant = manifeste["document_id"]
        lignes = extractions_par_document[identifiant]
        types = Counter(x["type_nomination"] for x in lignes)
        confiances = Counter(x["confiance_extraction"] for x in lignes)
        profil: dict[str, object] = {
            "document_id": identifiant,
            "annee": annee(identifiant),
            "bande_annee": bande_annee(annee(identifiant)),
            "pdf": str(Path(manifeste["pdf_source"]).resolve().relative_to(RACINE)),
            "methode_lecture": manifeste["methode_lecture"],
            "candidats": len(lignes),
            "individuelles": types["fonction_individuelle"],
            "conseils": types["conseil_administration"],
            "haute": confiances["haute"],
            "moyenne": confiances["moyenne"],
            "a_revoir": confiances["a_revoir"],
            "exclu_du_test": identifiant in exclus,
        }
        profil["score_risque"], profil["raisons_risque"] = risque(profil)
        profils.append(profil)

    disponibles = [x for x in profils if not x["exclu_du_test"]]
    if len(disponibles) < args.taille:
        raise ValueError("Pas assez de PDF hors des jeux déjà consultés.")

    selection: list[dict[str, object]] = []
    quotas = [
        ("ocr", 1, lambda x: x["methode_lecture"] != "pdfminer"),
        ("zero_candidat", 4, lambda x: x["candidats"] == 0),
        ("conseils_administration", 4, lambda x: x["conseils"] >= 5 and 5 <= x["candidats"] <= 80),
        ("champs_a_revoir", 3, lambda x: x["a_revoir"] > 0 and x["candidats"] <= 80),
        ("listes_longues", 2, lambda x: 75 <= x["candidats"] <= 130),
    ]
    for strate, quota, condition in quotas:
        disponibles_strate = [x for x in disponibles if x not in selection and condition(x)]
        for profil in choisir_equilibre(disponibles_strate, quota, args.graine + strate):
            if sum(int(x["candidats"]) for x in selection) + int(profil["candidats"]) > args.maximum_candidats:
                continue
            profil["strate_selection"] = strate
            selection.append(profil)

    standard = [
        x for x in disponibles
        if x not in selection and 5 <= int(x["candidats"]) <= 60 and not x["conseils"] and not x["a_revoir"]
    ]
    for profil in choisir_equilibre(standard, max(0, args.taille - len(selection)), args.graine + "standard"):
        if sum(int(x["candidats"]) for x in selection) + int(profil["candidats"]) > args.maximum_candidats:
            continue
        profil["strate_selection"] = "standard"
        selection.append(profil)
    if len(selection) < args.taille:
        reste = [x for x in disponibles if x not in selection]
        for profil in choisir_equilibre(reste, args.taille - len(selection), args.graine + "complement"):
            if int(profil["candidats"]) > 80 or sum(int(x["candidats"]) for x in selection) + int(profil["candidats"]) > args.maximum_candidats:
                continue
            profil["strate_selection"] = "complement"
            selection.append(profil)

    # Un test à 99 % doit contenir assez d'annonces ; compléter sans sortir du cadre fixé.
    reste = [x for x in disponibles if x not in selection]
    for profil in choisir_equilibre(reste, len(reste), args.graine + "seuil"):
        if sum(int(x["candidats"]) for x in selection) >= args.minimum_candidats:
            break
        if int(profil["candidats"]) > 80 or sum(int(x["candidats"]) for x in selection) + int(profil["candidats"]) > args.maximum_candidats:
            continue
        profil["strate_selection"] = "complement_pour_couverture"
        selection.append(profil)

    sorties_test = TEST / "textes"
    lignes_lot, lignes_reference, lignes_suivi, predictions_figees = [], [], [], []
    empreintes = empreinte()
    for ordre, profil in enumerate(selection, 1):
        identifiant = str(profil["document_id"])
        chemin_pdf = RACINE / str(profil["pdf"])
        # L'OCR a déjà été exécuté lors de la construction du manifeste corpus.
        # Pour l'interface, le texte natif est un confort de lecture ; le PDF
        # reste la source à contrôler, surtout lorsque ce texte est vide.
        from pdfminer.high_level import extract_text
        import pymupdf

        texte = extract_text(str(chemin_pdf)) or ""
        with pymupdf.open(chemin_pdf) as document:
            nombre_pages = len(document)
        chemin_texte = sorties_test / f"{chemin_pdf.stem}.txt"
        chemin_texte.parent.mkdir(parents=True, exist_ok=True)
        chemin_texte.write_text(texte, encoding="utf-8")
        lignes_lot.append({
            "ordre_relecture": ordre,
            **profil,
            "date_nom_fichier": pipeline.date_nom_fichier(chemin_pdf),
            "texte_pour_annotation": str(chemin_texte.relative_to(RACINE)),
            "nombre_pages": nombre_pages,
            "note_lecture": "Texte natif fourni pour aide à la lecture ; contrôler le PDF, notamment pour les scans.",
            "usage": "test_independant_verrouille",
            "empreinte_pipeline": empreintes["pipeline_extraction_nominations.ipynb"][:12],
            "empreinte_regles": empreintes["proposer_annotations.ipynb"][:12],
        })
        for candidat in extractions_par_document[identifiant]:
            predictions_figees.append(candidat)
            lignes_reference.append({
                champ: candidat.get(champ, "") for champ in COLONNES_REFERENCE
            } | {
                "statut_verification": "a_verifier",
                "mode_proposition": "pipeline_figee_test_independant",
            })
        lignes_suivi.append({
            "document_id": identifiant,
            "date_nom_fichier": pipeline.date_nom_fichier(chemin_pdf),
            "statut": "propositions_a_valider",
            "nombre_propositions": str(profil["candidats"]),
            "note": "Test indépendant verrouillé : lire intégralement le PDF, ajouter les annonces manquées et ne pas modifier les règles avant l'évaluation.",
        })

    ecrire(TEST / "lot_test_independant.csv", list(lignes_lot[0]), lignes_lot)
    ecrire(TEST / "reference_annotations.csv", COLONNES_REFERENCE, lignes_reference)
    ecrire(TEST / "reference_annotations_validees.csv", COLONNES_REFERENCE, [])
    ecrire(TEST / "annotations_consolidees_a_valider.csv", COLONNES_REFERENCE, lignes_reference)
    ecrire(TEST / "predictions_pipeline_figee.csv", list(extractions[0]), predictions_figees)
    ecrire(TEST / "suivi_annotations.csv", list(lignes_suivi[0]), lignes_suivi)
    (TEST / "validations_documents.json").write_text("{}\n", encoding="utf-8")
    (TEST / "empreinte_pipeline.json").write_text(json.dumps({
        "graine_selection": args.graine,
        "taille_pdf": len(selection),
        "candidats_pipeline": sum(int(x["candidats"]) for x in selection),
        "fichier_predictions_figees": "predictions_pipeline_figee.csv",
        "empreintes": empreintes,
        "consigne": "Ne pas modifier ces deux scripts avant la validation et l'évaluation du lot test.",
    }, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

    # Cette file sert uniquement au développement. Les PDF verrouillés pour le test en sont exclus.
    test_ids = {str(x["document_id"]) for x in selection}
    file_developpement = [x for x in profils if x["document_id"] not in test_ids]
    file_developpement.sort(key=lambda x: (-int(x["score_risque"]), str(x["document_id"])))
    ecrire(DOSSIER / "file_relecture_developpement_prioritaire.csv", list(file_developpement[0]), file_developpement)
    print(
        f"Test indépendant : {len(selection)} PDF, {sum(int(x['candidats']) for x in selection)} candidats. "
        f"Répertoire : {TEST}. File de développement : evaluation_extraction/file_relecture_developpement_prioritaire.csv"
    )


## Exécution manuelle

Vérifier les chemins et les fichiers de sortie avant toute exécution.

In [ ]:
# main()